# Judge SD pre-flight
Prints the OmniContext scorer config and checks baseline alignment.

In [ ]:
# mount Drive, clone OmniGen2 (skip if already cloned)
import os, subprocess
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/VectorSpaceLab/OmniGen2", REPO],
                   check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
OMNI = os.path.join(REPO, "omnicontext")

Mounted at /content/drive


In [ ]:
# data paths
DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
TEST     = f"{DRIVE}/test.jsonl"
BASELINE = f"{DRIVE}/scores/test__baseline.jsonl"

In [ ]:
# print the scorer config from source; compute baseline alignment
import json, re

def src(f):
    p = os.path.join(OMNI, f)
    return open(p, encoding="utf-8").read() if os.path.exists(p) else ""

s = src("openai_util.py")
m = re.search(r"payload\s*=\s*\{.*?\n\s*\}", s, re.S)
print("----- openai_util.py : request payload -----")
print(m.group(0) if m else "payload not found")

s = src("omnicontext_score.py")
i = s.find("for try_idx")
k = s.find(":", s.find("except Exception", i)) + 1 if i >= 0 else -1
print("\n----- omnicontext_score.py : retry loop -----")
print(s[i:k] if i >= 0 else "retry loop not found")

s = src("calculate_statistics.py")
print("\n----- calculate_statistics.py : Overall -----")
print(next((l.strip() for l in s.splitlines() if "math.sqrt(prompt_following_score" in l), "not found"))

def load(p): return [json.loads(x) for x in open(p, encoding="utf-8")]
print("\n----- baseline alignment -----")
if os.path.exists(TEST) and os.path.exists(BASELINE):
    test, base = load(TEST), load(BASELINE)
    uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
    tu = [uid(r) for r in test]
    aligned = sum(1 for b in base if 0 <= b["i"] < len(test) and b["uid"] == tu[b["i"]])
    print("test=%d  baseline=%d  i<->uid aligned=%d/%d" % (len(test), len(base), aligned, len(base)))
else:
    print("missing file:", "" if os.path.exists(TEST) else TEST, "" if os.path.exists(BASELINE) else BASELINE)

----- openai_util.py : request payload -----
payload = {
        "model": "gpt-4.1",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1400,
    }

----- omnicontext_score.py : retry loop -----
for try_idx in range(max_tries):
            try:
                PF_prompt = self.prompt_generator(instruction, task_type="prompt_following")
                SC_prompt = self.prompt_generator(instruction, task_type="subject_consistency", with_scene=with_scene)

                PF_results = ask_gpt4o(input_image_paths, PF_prompt, self.openai_url, self.openai_key)
                SC_results = ask_gpt4o(input_image_paths, SC_prompt, self.openai_url, self.openai_key)

                PF_scores = mllm_output_to_dict(PF_results)
                SC_scores = mllm_output_to_dict(SC_results)

                if PF_scores == "rate_limit_exceeded" or SC_scores == "rate_limit_exceeded":
                    raise Exception("rate_limit_exceeded")
            except Exception as 